# 🥚 Egg Detection System using Deep Learning and YOLOv11n

Detect and count eggs in images, videos and live camera streams with a **YOLOv11n** (CNN-based) object detector fine-tuned on a Kaggle dataset.

## 1. Install required packages

In [ ]:
%pip install -q -r ../requirements.txt

## 2. Imports and configuration

In [ ]:
import sys
from pathlib import Path

# make the project's src/ folder importable
SRC_DIR = Path.cwd().parent / "src" if Path.cwd().name == "notebooks" else Path.cwd() / "src"
sys.path.insert(0, str(SRC_DIR))

import cv2
import matplotlib.pyplot as plt

import utils, train, evaluate, predict

# ---------------- user settings ----------------
KAGGLE_DATASET = utils.KAGGLE_DATASET   # or write e.g. "owner/egg-detection"
EPOCHS = 50
IMG_SIZE = 640
BATCH_SIZE = 16
LEARNING_RATE = 0.01
PATIENCE = 15                            # early stopping
CONFIDENCE_THRESHOLD = 0.5               # only detections >= this are displayed
EGG_CLASS_IDS = None                     # None = every annotated object is an egg
# -----------------------------------------------

DEVICE = utils.get_device()              # GPU if CUDA is available, otherwise CPU
print("Project root:", utils.PROJECT_ROOT)

## 3. Kaggle API setup (secure)

1. On kaggle.com open **Settings → API → Create New Token** to download `kaggle.json`.
2. Move it to `~/.kaggle/kaggle.json` and run `chmod 600 ~/.kaggle/kaggle.json` (Windows: `C:\Users\<you>\.kaggle\`).
3. Or set the environment variables `KAGGLE_USERNAME` and `KAGGLE_KEY`.

> **Never** commit `kaggle.json` to GitHub or paste your key into a cell. The `.gitignore` already excludes it.

To discover a suitable dataset, run the optional search below and copy the `owner/slug` into `KAGGLE_DATASET` above.

In [ ]:
# Optional: list Kaggle datasets that match a search term (requires credentials)
# utils.search_kaggle("egg detection")

## 4. Download the dataset programmatically

In [ ]:
try:
    utils.download_kaggle_dataset(KAGGLE_DATASET)
except (FileNotFoundError, ValueError, ImportError) as err:
    print("[ERROR]", err)
    print("If you already have a YOLO-format egg dataset, copy it into", utils.RAW_DIR, "and continue.")

## 5. Prepare the YOLO-format dataset and create `data.yaml`

In [ ]:
counts = utils.prepare_dataset(utils.RAW_DIR, utils.DATASET_DIR, keep_class_ids=EGG_CLASS_IDS)
print(open(utils.DATA_YAML).read())